In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# 1. Configuração do DataFrame de Features (caso esteja rodando em um arquivo isolado)
dados_ficticios = {
    'danceability': [0.8, 0.5, 0.6, 0.7, 0.4],
    'energy': [0.9, 0.4, 0.7, 0.6, 0.5],
    'loudness': [-5.0, -12.0, -8.0, -6.5, -10.0],
    'mode': [1, 0, 1, 1, 0],
    'speechiness': [0.05, 0.03, 0.08, 0.04, 0.06],
    'acousticness': [0.1, 0.8, 0.3, 0.2, 0.5],
    'instrumentalness': [0.0, 0.5, 0.01, 0.0, 0.02],
    'liveness': [0.2, 0.1, 0.15, 0.25, 0.18],
    'valence': [0.85, 0.3, 0.6, 0.7, 0.4],
    'tempo': [125.5, 90.0, 110.0, 118.0, 95.0]
}

df = pd.DataFrame(dados_ficticios)
audio_cols = list(df.columns)

# Normalização Min-Max para padronizar a escala entre [0, 1]
scaler = MinMaxScaler()
df_features = df.copy()
df_features[audio_cols] = scaler.fit_transform(df_features[audio_cols])

# 2. Definição da Classe do Ambiente de Aprendizado por Reforço
class MusicEnvironment:
    def __init__(self, df_features):
        self.df = df_features
        self.current_idx = 0
        
    def reset(self):
        # Reinicia o ambiente escolhendo uma música aleatória como ponto de partida (Ponto A)
        self.current_idx = np.random.randint(0, len(self.df))
        return self.df.iloc[self.current_idx].values
        
    def step(self, next_idx):
        # Isola os vetores da música atual e da próxima escolha
        current_vector = self.df.iloc[self.current_idx].values.reshape(1, -1)
        next_vector = self.df.iloc[next_idx].values.reshape(1, -1)
        
        # Calcula a compatibilidade sonora via Similaridade por Cosseno
        similarity = cosine_similarity(current_vector, next_vector)[0][0]
        
        # Função de Recompensa baseada na regra de negócio (transição fluida vs. skip)
        if similarity < 0.4:
            reward = -1.0  # Punição: transição muito brusca (provoca skip)
            done = True
        elif 0.6 <= similarity <= 0.95:
            reward = 1.0   # Recompensa: harmonia mantida com sucesso
            done = False
        else:
            reward = 0.1   # Neutro: aceitável, mas fora da zona ideal
            done = False
            
        # Atualiza o ponteiro de reprodução
        self.current_idx = next_idx
        next_state = self.df.iloc[self.current_idx].values
        
        return next_state, reward, done

# 3. Teste prático do Ambiente e da Função de Recompensa
env = MusicEnvironment(df_features)
estado_atual = env.reset()
print("Música inicial (Ponto A) selecionada com sucesso.")

# Simula o agente escolhendo o índice 2 do dataset para a próxima faixa
proximo_estado, recompensa, fim_da_sessao = env.step(2)

print(f"Recompensa calculada pelo ambiente: {recompensa}")
print(f"A sessão terminou (ocorreu skip)? {fim_da_sessao}")

Música inicial (Ponto A) selecionada com sucesso.
Recompensa calculada pelo ambiente: 1.0
A sessão terminou (ocorreu skip)? False


In [2]:
# Inicializa o ambiente com o DataFrame tratado
env = MusicEnvironment(df_features)

# Reseta o ambiente para pegar a primeira música aleatória (Ponto A)
estado_atual = env.reset()
print("Música inicial (Ponto A) selecionada com sucesso. Vetor:", estado_atual)

# Simula o agente escolhendo o índice 2 do dataset para a próxima faixa
proximo_estado, recompensa, fim_da_sessao = env.step(2)

print(f"\nRecompensa calculada pelo ambiente: {recompensa}")
print(f"A sessão terminou (ocorreu skip)? {fim_da_sessao}")

Música inicial (Ponto A) selecionada com sucesso. Vetor: [0.25 0.   0.   0.   0.   1.   1.   0.   0.   0.  ]

Recompensa calculada pelo ambiente: -1.0
A sessão terminou (ocorreu skip)? True
